In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
 
print("All tools loaded. Ready to build an auto-router.")


All tools loaded. Ready to build an auto-router.


In [5]:
import numpy as np
import pandas as pd
import os

np.random.seed(42)

network_templates = [
    "There is no network signal in {place}",
    "The network is very slow in {place}",
    "Calls keep dropping in {place}",
    "I have poor signal coverage in {place}",
    "Mobile internet is not working in {place}",
    "The 5G connection is unstable in {place}"
]

billing_templates = [
    "My bill is too high this month",
    "The invoice is wrong can you explain",
    "I was charged extra on my bill",
    "There is an incorrect charge on my invoice",
    "Please explain the charges on my bill",
    "My monthly bill amount is incorrect"
]

sim_templates = [
    "My new SIM is not activating please help",
    "I cannot activate my SIM",
    "My eSIM activation failed",
    "The new SIM card is not working",
    "I need help activating my eSIM",
    "SIM activation is taking too long"
]

plan_templates = [
    "I would like to change my monthly plan",
    "I want to upgrade to a bigger data plan",
    "What plans do you have for more data",
    "I would like to switch to postpaid",
    "Can I change my current plan",
    "I want a cheaper monthly plan"
]

places = ["Muscat", "Salalah", "Sohar", "Nizwa", "Sur"]

rows = []

def fill_place(text):
    return text.replace("{place}", np.random.choice(places))

for _ in range(70):
    rows.append((fill_place(np.random.choice(network_templates)), "Network"))

for _ in range(64):
    rows.append((np.random.choice(billing_templates), "Billing"))

for _ in range(53):
    rows.append((np.random.choice(sim_templates), "SIM_Activation"))

for _ in range(58):
    rows.append((np.random.choice(plan_templates), "Plan_Change"))

df = pd.DataFrame(rows, columns=["ticket_text", "team"])

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

df.insert(0, "ticket_id", [f"TKT{70000+i}" for i in range(len(df))])

# Save CSV
file_path = os.path.join(os.getcwd(), "omantel_support_tickets.csv")
df.to_csv(file_path, index=False)

print("CSV created successfully!")
print("Location:", file_path)
print("Number of tickets:", len(df))
print("\nTeam distribution:")
print(df["team"].value_counts())

CSV created successfully!
Location: C:\Users\Acer\OneDrive\Desktop\AI & ML\Ticket Routing - Ex 11\omantel_support_tickets.csv
Number of tickets: 245

Team distribution:
team
Network           70
Billing           64
Plan_Change       58
SIM_Activation    53
Name: count, dtype: int64


In [6]:
df.head(10)

,ticket_id,ticket_text,team
0,TKT70000,The network is very slow in Muscat,Network
1,TKT70001,The 5G connection is unstable in Sur,Network
2,TKT70002,I need help activating my eSIM,SIM_Activation
3,TKT70003,I would like to switch to postpaid,Plan_Change
4,TKT70004,Can I change my current plan,Plan_Change
5,TKT70005,My new SIM is not activating please help,SIM_Activation
6,TKT70006,I would like to change my monthly plan,Plan_Change
7,TKT70007,There is an incorrect charge on my invoice,Billing
8,TKT70008,I have poor signal coverage in Sur,Network
9,TKT70009,My bill is too high this month,Billing


In [7]:
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df["ticket_text"])
y = df["team"]
print(f"Each ticket is now {X.shape[1]} number-columns the machine can read.")

Each ticket is now 80 number-columns the machine can read.


In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)
print(f"Training tickets: {X_train.shape[0]}   Exam tickets: {X_test.shape[0]}")

Training tickets: 183   Exam tickets: 62


In [9]:
router = LogisticRegression(max_iter=1000)
router.fit(X_train, y_train)
print("Router trained.")

Router trained.


In [10]:
preds = router.predict(X_test)
acc = accuracy_score(y_test, preds) * 100
print(f"Routing accuracy: {acc:.1f}%")
print(classification_report(y_test, preds))

Routing accuracy: 100.0%
                precision    recall  f1-score   support

       Billing       1.00      1.00      1.00        16
       Network       1.00      1.00      1.00        18
   Plan_Change       1.00      1.00      1.00        15
SIM_Activation       1.00      1.00      1.00        13

      accuracy                           1.00        62
     macro avg       1.00      1.00      1.00        62
  weighted avg       1.00      1.00      1.00        62



In [11]:
labels = sorted(df["team"].unique())
cm = confusion_matrix(y_test, preds, labels=labels)
print(pd.DataFrame(cm, index=labels, columns=labels).to_string())

                Billing  Network  Plan_Change  SIM_Activation
Billing              16        0            0               0
Network               0       18            0               0
Plan_Change           0        0           15               0
SIM_Activation        0        0            0              13


In [12]:
hard = [
    "My new plan shows the wrong price on my bill",
    "I changed my plan but now the internet does not work",
    "The offer charged me but my SIM is not active",
    "Slow internet and my bill went up too",
]
probs = router.predict_proba(vectorizer.transform(hard))
classes = router.classes_
for t, pr in zip(hard, probs):
    top = sorted(zip(classes, pr), key=lambda x: -x[1])[:2]
    print(f"{t}")
    print(f"   -> {top[0][0]} ({top[0][1]*100:.0f}%) | 2nd guess: {top[1][0]} ({top[1][1]*100:.0f}%)\n")

My new plan shows the wrong price on my bill
   -> Billing (62%) | 2nd guess: SIM_Activation (18%)

I changed my plan but now the internet does not work
   -> Network (31%) | 2nd guess: Plan_Change (28%)

The offer charged me but my SIM is not active
   -> SIM_Activation (55%) | 2nd guess: Billing (24%)

Slow internet and my bill went up too
   -> Billing (49%) | 2nd guess: Network (27%)



In [13]:
new_tickets = [
    "My bill is too high this month please check",
    "There is no network signal in Nizwa",
    "I cannot activate my new eSIM",
    "I want to upgrade to a bigger data plan",
]
routes = router.predict(vectorizer.transform(new_tickets))
print("LIVE ROUTING:")
for t, r in zip(new_tickets, routes):
    print(f"   -> [{r:15}]  {t}")

LIVE ROUTING:
   -> [Billing        ]  My bill is too high this month please check
   -> [Network        ]  There is no network signal in Nizwa
   -> [SIM_Activation ]  I cannot activate my new eSIM
   -> [Plan_Change    ]  I want to upgrade to a bigger data plan
